# Race to 10 — 2026 NCAA Tournament Analysis

Uses PBP data + CBBD sportsbook lines to compute each team's race-to-10 win rates,
stratified by implied spread, then projects all 32 first-round tournament matchups.

**Outputs:**
- `race_to_10_team_stats.csv` — all D1 teams, win rates by spread bucket
- `race_to_10_actual_vs_expected.csv` — per-team actual vs model-expected win rates
- `race_to_10_tournament.csv` — 32 matchup projections with EV picks

## 0. Setup

In [ ]:
import getpass
import glob
import os
import warnings

import cbbd
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit
from scipy.special import expit as sigmoid

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.3f}'.format)

print('Libraries loaded ✓')

## 1. API Key & Config

In [ ]:
# Enter your CBBD API key (input is hidden)
API_KEY = getpass.getpass('Enter your CBBD API key: ')

In [ ]:
# ── Paths & settings ─────────────────────────────────────────────────
SEASON      = 2026
PBP_DIR     = 'cbbd_data/pbp_flat'
GAMES_DIR   = 'cbbd_data/games'
LINES_CACHE = 'cbbd_data/lines/2026_lines.csv'
RATINGS_CSV = 'team_ratings_cache.csv'
TARGET      = 10   # "race to N" threshold

print(f'Config set. Target = {TARGET} points | Season = {SEASON}')

## 2. Fetch Sportsbook Lines

Lines are cached after first fetch so re-running is fast.  
`spread` is from the **home team's perspective**: negative = home favored, positive = away favored.

In [ ]:
def _parse_lines_response(raw) -> list:
    rows = []
    for gl in raw:
        spreads  = [l.spread         for l in gl.lines if l.spread         is not None]
        ous      = [l.over_under     for l in gl.lines if l.over_under     is not None]
        home_mls = [l.home_moneyline for l in gl.lines if l.home_moneyline is not None]
        away_mls = [l.away_moneyline for l in gl.lines if l.away_moneyline is not None]
        rows.append({
            'gameId':        gl.game_id,
            'homeTeam':      gl.home_team,
            'awayTeam':      gl.away_team,
            'startDate':     str(gl.start_date),
            'homeScore':     gl.home_score,
            'awayScore':     gl.away_score,
            'spread':        float(np.mean(spreads))  if spreads  else np.nan,
            'overUnder':     float(np.mean(ous))      if ous      else np.nan,
            'homeMoneyline': float(np.mean(home_mls)) if home_mls else np.nan,
            'awayMoneyline': float(np.mean(away_mls)) if away_mls else np.nan,
            'n_providers':   len(gl.lines),
        })
    return rows


def fetch_lines(api_key: str, season: int, cache_path: str, games_dir: str) -> pd.DataFrame:
    if os.path.exists(cache_path):
        print(f'Loading from cache: {cache_path}')
        df = pd.read_csv(cache_path)
        print(f'  {len(df):,} games with lines')
        return df

    # Iterate by team — avoids the ~3000 record cap on the bulk endpoint.
    # team= returns both home and away games; seen_ids deduplicates.
    game_files = sorted(glob.glob(f'{games_dir}/*.csv'))
    all_games  = pd.concat([pd.read_csv(f, usecols=['homeTeam']) for f in game_files],
                           ignore_index=True)
    teams = sorted(all_games['homeTeam'].dropna().unique())
    print(f'Fetching {season} lines by team ({len(teams)} teams)...')

    configuration = cbbd.Configuration(access_token=api_key)
    all_rows = []
    seen_ids = set()

    with cbbd.ApiClient(configuration) as api_client:
        lines_api = cbbd.LinesApi(api_client)
        for i, team in enumerate(teams, 1):
            if i % 50 == 0 or i == len(teams):
                print(f'  {i}/{len(teams)} teams — {len(seen_ids):,} unique games so far')
            try:
                raw = lines_api.get_lines(season=season, team=team)
                for row in _parse_lines_response(raw):
                    if row['gameId'] not in seen_ids:
                        seen_ids.add(row['gameId'])
                        all_rows.append(row)
            except Exception as e:
                print(f'  ⚠ {team}: {e}')
                continue

    df = pd.DataFrame(all_rows)
    os.makedirs(os.path.dirname(cache_path), exist_ok=True)
    df.to_csv(cache_path, index=False)
    print(f'Saved {len(df):,} games → {cache_path}')
    return df


lines = fetch_lines(API_KEY, SEASON, LINES_CACHE, GAMES_DIR)
lines.head(3)

### Quick sanity check: does negative spread = home favored?
If `homeMoneyline < -100` and `spread < 0` → convention is correct.

In [ ]:
sample = lines.dropna(subset=['spread', 'homeMoneyline'])

home_fav_ml    = sample['homeMoneyline'] < -110
spread_neg     = sample['spread'] < 0
aligns         = (home_fav_ml & spread_neg).sum()
total_home_fav = home_fav_ml.sum()

print(f"Home ML favorites (< -110): {total_home_fav:,}")
print(f"  spread < 0 (home fav):    {aligns:,}  ({aligns/total_home_fav:.1%})")
print(f"  spread > 0 (home dog??):  {(home_fav_ml & ~spread_neg).sum():,}  ({(home_fav_ml & ~spread_neg).mean():.1%})")

print("\n--- Sample misaligned rows (home is ML fav but spread is positive) ---")
misaligned = sample[home_fav_ml & ~spread_neg][
    ['homeTeam','awayTeam','homeMoneyline','awayMoneyline','spread']
].head(15)
print(misaligned.to_string(index=False))

print("\n--- Sample aligned rows (home is ML fav and spread is negative) ---")
aligned = sample[home_fav_ml & spread_neg][
    ['homeTeam','awayTeam','homeMoneyline','awayMoneyline','spread']
].head(10)
print(aligned.to_string(index=False))

## 3. Process PBP — Race-to-10 Per Game

In [ ]:
def elapsed_secs(period, secs_remaining):
    return (period - 1) * 1200 + (1200 - secs_remaining)


def process_pbp(pbp_dir: str, target: int = 10) -> pd.DataFrame:
    files = sorted(glob.glob(f'{pbp_dir}/*.csv'))
    print(f'Loading {len(files)} PBP files...')

    chunks = []
    for f in files:
        chunks.append(pd.read_csv(f, usecols=[
            'gameId', 'period', 'secondsRemaining',
            'homeScore', 'awayScore', 'scoringPlay', 'isHomeTeam',
        ]))
    pbp = pd.concat(chunks, ignore_index=True)
    print(f'  {len(pbp):,} play rows')

    pbp['elapsed'] = elapsed_secs(pbp['period'], pbp['secondsRemaining'])
    pbp = pbp.sort_values(['gameId', 'elapsed'])

    records = []
    for game_id, grp in pbp.groupby('gameId'):
        grp = grp.reset_index(drop=True)

        # ── Race to target ─────────────────────────────────────────────────────
        home_hit = grp[grp['homeScore'] >= target]
        away_hit = grp[grp['awayScore'] >= target]
        if home_hit.empty or away_hit.empty:
            continue

        home_t = home_hit.iloc[0]['elapsed']
        away_t = away_hit.iloc[0]['elapsed']
        home_wins = home_t < away_t

        # ── First scorer ───────────────────────────────────────────────────────
        scoring = grp[grp['scoringPlay'] == True]
        if scoring.empty:
            continue
        first_is_home = scoring.iloc[0]['isHomeTeam']
        home_scored_first = bool(first_is_home) if pd.notna(first_is_home) else None

        # ── Max early lead (before either team hits target) ────────────────────
        early = grp[(grp['homeScore'] < target) & (grp['awayScore'] < target)]
        if not early.empty:
            lead = early['homeScore'] - early['awayScore']
            max_home_lead = int(lead.max())
            max_away_lead = int((-lead).max())
        else:
            max_home_lead = max_away_lead = 0

        # ── Blitz: reach target before opponent reaches 5 ──────────────────────
        away_5 = grp[grp['awayScore'] >= 5]
        home_5 = grp[grp['homeScore'] >= 5]
        away_5_t = away_5.iloc[0]['elapsed'] if not away_5.empty else np.inf
        home_5_t = home_5.iloc[0]['elapsed'] if not home_5.empty else np.inf
        home_blitz = bool(home_t < away_5_t)
        away_blitz = bool(away_t < home_5_t)

        records.append({
            'gameId':            game_id,
            'home_t10':          home_t,
            'away_t10':          away_t,
            'home_wins_race':    home_wins,
            'home_scored_first': home_scored_first,
            'max_home_lead':     max_home_lead,
            'max_away_lead':     max_away_lead,
            'home_blitz':        home_blitz,
            'away_blitz':        away_blitz,
        })

    out = pd.DataFrame(records)
    print(f'  Race-to-{target} computed for {len(out):,} games')
    return out


race_df = process_pbp(PBP_DIR, TARGET)
race_df.head(3)

## 4. Load Games Metadata & Merge

In [ ]:
# Load games for home/away team names and neutral site flag
game_files = sorted(glob.glob(f'{GAMES_DIR}/*.csv'))
games = pd.concat([pd.read_csv(f) for f in game_files], ignore_index=True)
games = games[games['status'] == 'final'].drop_duplicates('id')
games = games[['id','homeTeam','awayTeam','neutralSite']].rename(columns={'id':'gameId'})
print(f'Loaded {len(games):,} completed games')

# Merge race data + games + lines
df = race_df.merge(games, on='gameId', how='inner')

lines_slim = lines[['gameId','spread']].dropna(subset=['spread'])
df = df.merge(lines_slim, on='gameId', how='left')

# Convert API spread to team-perspective spread
# API spread: negative = home favored
# team_spread: positive = team is favored
df['home_spread'] = -df['spread']   # flip: home fav → positive
df['away_spread'] =  df['spread']   # away fav when positive

print(f'Games with sportsbook spread: {df["spread"].notna().sum():,} / {len(df):,}')
df.head(3)

## 5. Build Team-Level Records

Two rows per game — one from home team's POV, one from away.

In [ ]:
home_rows = pd.DataFrame({
    'gameId':            df['gameId'],
    'team':              df['homeTeam'],
    'opponent':          df['awayTeam'],
    'is_home':           True,
    'neutral':           df['neutralSite'],
    'team_spread':       df['home_spread'],
    'team_t10':          df['home_t10'],
    'opp_t10':           df['away_t10'],
    'team_wins_race':    df['home_wins_race'].astype(float),
    'team_scored_first': df['home_scored_first'],
    'max_team_lead':     df['max_home_lead'],
    'team_blitz':        df['home_blitz'].astype(float),
})

away_rows = pd.DataFrame({
    'gameId':            df['gameId'],
    'team':              df['awayTeam'],
    'opponent':          df['homeTeam'],
    'is_home':           False,
    'neutral':           df['neutralSite'],
    'team_spread':       df['away_spread'],
    'team_t10':          df['away_t10'],
    'opp_t10':           df['home_t10'],
    'team_wins_race':    (~df['home_wins_race']).astype(float),
    'team_scored_first': df['home_scored_first'].map(
                             lambda x: (not x) if pd.notna(x) else None),
    'max_team_lead':     df['max_away_lead'],
    'team_blitz':        df['away_blitz'].astype(float),
})

tg = pd.concat([home_rows, away_rows], ignore_index=True)
print(f'{len(tg):,} team-game records')
print(f'  With spread: {tg["team_spread"].notna().sum():,}')
tg.head(4)

## 6. Aggregate Per-Team Stats (All D1)

Spread buckets:  
`Fav_10p` = 10+ pt fav | `Fav_5_10` = 5–10 pt fav | `Close` = within 5  
`Dog_5_10` = 5–10 pt dog | `Dog_10p` = 10+ pt dog

In [ ]:
BUCKETS = [
    ('Fav_10p',   10,  999),
    ('Fav_5_10',   5,   10),
    ('Close',     -5,    5),
    ('Dog_5_10', -10,   -5),
    ('Dog_10p',  -999, -10),
]

def spread_bucket(s):
    if pd.isna(s):
        return 'No_line'
    for name, lo, hi in BUCKETS:
        if lo <= s < hi:
            return name
    return 'Fav_10p' if s >= 10 else 'Dog_10p'

tg['bucket'] = tg['team_spread'].apply(spread_bucket)

# Base aggregation
stats = tg.groupby('team').agg(
    games              = ('gameId',          'count'),
    race_win_pct       = ('team_wins_race',  'mean'),
    avg_t10_secs       = ('team_t10',        'mean'),
    scored_first_pct   = ('team_scored_first','mean'),
    avg_max_lead_early = ('max_team_lead',   'mean'),
    blitz_rate         = ('team_blitz',      'mean'),
).round(3)

# Per-bucket win rate and sample size
for bucket_name, _, _ in BUCKETS:
    sub = tg[tg['bucket'] == bucket_name]
    bwr = sub.groupby('team')['team_wins_race'].agg(
        wr='mean', n='count'
    )
    stats[f'wr_{bucket_name}'] = bwr['wr'].round(3)
    stats[f'n_{bucket_name}']  = bwr['n']

stats = stats.reset_index()
print(f'Aggregated stats for {len(stats):,} teams')

# Attach ratings
if os.path.exists(RATINGS_CSV):
    ratings = pd.read_csv(RATINGS_CSV)[['team','adj_net','adj_ortg','adj_drtg','tempo']]
    stats = stats.merge(ratings, on='team', how='left')
    print('Attached adj_net ratings ✓')

stats.head(5)

## 7. Fit Logistic Model

```
P(team wins race to 10) = sigmoid(k × team_spread + b)
```
Trained on all games with sportsbook spread data.

In [ ]:
model_df = tg.dropna(subset=['team_spread','team_wins_race'])
print(f'Model training set: {len(model_df):,} team-game observations')

X = model_df['team_spread'].values.astype(float)
y = model_df['team_wins_race'].values.astype(float)

def logistic(x, k, b):
    return sigmoid(k * x + b)

popt, _ = curve_fit(logistic, X, y, p0=[0.05, 0.0], maxfev=20000)
k, b = popt

print(f'\nFit: P(win race) = sigmoid({k:.4f} × spread + {b:.4f})')
print(f'\n--- Implied probabilities ---')
for pts in [-10, -5, -3, -1, 0, 1, 3, 5, 10]:
    p = float(sigmoid(k * pts + b))
    label = f'{abs(pts):2d}pt {"dog" if pts < 0 else ("fav" if pts > 0 else "pick"):>4}'
    bar = '█' * int(p * 40)
    print(f'  {label}: {p:.1%}  {bar}')

## 8. Actual vs Expected Race-to-10 Win Rate (Full Season)

For every game with a sportsbook spread, the model predicts each team's probability of winning the race to 10.  
Summing those predictions gives an **expected win total** — compare to actual wins to find teams that  
consistently out- or under-shoot their implied probability.

- **Positive diff** = team wins the race to 10 more than the spread says they should (fast starters)  
- **Negative diff** = team consistently loses the race to 10 even when favored (slow starters)

> Note: `diff` here is in-sample (model trained on same data). Use section 9 OOS diffs for validation.

In [ ]:
# Games with sportsbook spread only — can't compute expected without it
with_line = tg.dropna(subset=['team_spread']).copy()
with_line['expected_p'] = sigmoid(k * with_line['team_spread'].values + b)

ae = with_line.groupby('team').agg(
    n_games       = ('team_wins_race', 'count'),
    actual_wins   = ('team_wins_race', 'sum'),
    expected_wins = ('expected_p',     'sum'),
    actual_wr     = ('team_wins_race', 'mean'),
    expected_wr   = ('expected_p',     'mean'),
).round(3)

ae['diff']          = (ae['actual_wr'] - ae['expected_wr']).round(3)
ae['actual_wins']   = ae['actual_wins'].astype(int)
ae['expected_wins'] = ae['expected_wins'].round(1)
ae = ae.reset_index()

# Attach tournament flag and adj_net
# (in_tournament added in section 10 — re-merge after that cell if needed)
if 'in_tournament' in stats.columns:
    ae = ae.merge(stats[['team','in_tournament','adj_net']], on='team', how='left')

ae.to_csv('race_to_10_actual_vs_expected.csv', index=False)
print(f'Saved race_to_10_actual_vs_expected.csv ({len(ae):,} teams, {with_line["gameId"].nunique():,} games with lines)\n')

print('=== BIGGEST OVERPERFORMERS (actual > expected) ===')
over = ae[ae['n_games'] >= 10].sort_values('diff', ascending=False)
print(over[['team','n_games','actual_wins','expected_wins','actual_wr','expected_wr','diff']]
      .head(15).to_string(index=False))

print('\n=== BIGGEST UNDERPERFORMERS (actual < expected) ===')
print(over.sort_values('diff')
      [['team','n_games','actual_wins','expected_wins','actual_wr','expected_wr','diff']]
      .head(15).to_string(index=False))

## 9. Out-of-Sample Validation — 90/10 SRS Split

Fit the logistic model on a random 90% of **games**, evaluate on the held-out 10%.  
Game-level metrics (Brier score, calibration) tell us whether the model's predicted probabilities are reliable.

In [ ]:
SEED = 42
rng = np.random.default_rng(SEED)
with_line = tg.dropna(subset=['team_spread']).copy()

# Split on unique games (not rows) so home+away rows stay together
game_ids = with_line['gameId'].unique()
n_test   = max(1, int(len(game_ids) * 0.10))
test_ids = set(rng.choice(game_ids, size=n_test, replace=False))

train = with_line[~with_line['gameId'].isin(test_ids)]
test  = with_line[ with_line['gameId'].isin(test_ids)]

print(f'Train: {len(train):,} team-game rows ({train["gameId"].nunique():,} games)')
print(f'Test:  {len(test):,}  team-game rows ({test["gameId"].nunique():,} games)')

# ── Fit on train only ─────────────────────────────────────────────────────────
X_tr = train['team_spread'].values.astype(float)
y_tr = train['team_wins_race'].values.astype(float)
popt_oos, _ = curve_fit(logistic, X_tr, y_tr, p0=[0.05, 0.0], maxfev=20000)
k_oos, b_oos = popt_oos
print(f'\nOOS model:  sigmoid({k_oos:.4f} × spread + {b_oos:.4f})')
print(f'Full model: sigmoid({k:.4f} × spread + {b:.4f})')

# ── Game-level predictions on holdout ─────────────────────────────────────────
test = test.copy()
test['pred'] = sigmoid(k_oos * test['team_spread'].values + b_oos)
y_true = test['team_wins_race'].values
y_pred = test['pred'].values

brier          = float(np.mean((y_true - y_pred) ** 2))
baseline_brier = float(np.mean((y_true - y_true.mean()) ** 2))
print(f'\n── Game-level OOS metrics ({len(test):,} team-game rows) ──')
print(f'Brier score:          {brier:.4f}')
print(f'Baseline Brier (mean):{baseline_brier:.4f}')
print(f'Brier skill score:    {1 - brier/baseline_brier:.3f}  (0=no skill, 1=perfect)')

# ── Calibration bins ──────────────────────────────────────────────────────────
bin_edges = [0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65]
bin_labels = [f'{int(bin_edges[i]*100)}-{int(bin_edges[i+1]*100)}%'
              for i in range(len(bin_edges) - 1)]

test['pred_bin'] = pd.cut(test['pred'], bins=bin_edges, labels=bin_labels)
cal = test.groupby('pred_bin', observed=True).agg(
    n         = ('team_wins_race', 'count'),
    actual    = ('team_wins_race', 'mean'),
    predicted = ('pred',           'mean'),
).dropna()
cal['diff'] = (cal['actual'] - cal['predicted']).round(3)

print('\n── Calibration (predicted vs actual win rate per bin) ──')
print(f"{'Bin':>10}  {'n':>5}  {'predicted':>10}  {'actual':>8}  {'diff':>6}")
for bin_label, row in cal.iterrows():
    flag = ' ⚠' if abs(row['diff']) > 0.05 else ''
    print(f"{str(bin_label):>10}  {int(row['n']):>5}  "
          f"{row['predicted']:>10.1%}  {row['actual']:>8.1%}  {row['diff']:>+6.3f}{flag}")

# Store for downstream use
k_final, b_final = k_oos, b_oos

# Per-team OOS diff (weak secondary signal — small n per team)
ae_oos = test.groupby('team').agg(
    n_games_oos    = ('team_wins_race', 'count'),
    actual_wr_oos  = ('team_wins_race', 'mean'),
    expected_wr_oos= ('pred',           'mean'),
).round(3)
ae_oos['diff_oos'] = (ae_oos['actual_wr_oos'] - ae_oos['expected_wr_oos']).round(3)
ae_oos = ae_oos.reset_index()
compare = ae_oos

## 10. Flag Tournament Teams & Save Team Stats

In [ ]:
TOURNAMENT_MATCHUPS = [
    # EAST
    ('Duke',              1,  'Siena',              16, 'East'),
    ('Ohio St.',          8,  'TCU',                 9, 'East'),
    ("St. John's",        5,  'Northern Iowa',      12, 'East'),
    ('Kansas',            4,  'Cal Baptist',        13, 'East'),
    ('Louisville',        6,  'South Florida',      11, 'East'),
    ('Michigan St.',      3,  'North Dakota St.',   14, 'East'),
    ('UCLA',              7,  'UCF',                10, 'East'),
    ('UConn',             2,  'Furman',             15, 'East'),
    # SOUTH
    ('Florida',           1,  'Lehigh',             16, 'South'),
    ('Clemson',           8,  'Iowa',                9, 'South'),
    ('Vanderbilt',        5,  'McNeese',            12, 'South'),
    ('Nebraska',          4,  'Troy',               13, 'South'),
    ('North Carolina',    6,  'VCU',                11, 'South'),
    ('Illinois',          3,  'Penn',               14, 'South'),
    ("Saint Mary's",      7,  'Texas A&M',          10, 'South'),
    ('Houston',           2,  'Idaho',              15, 'South'),
    # WEST
    ('Arizona',           1,  'Long Island University', 16, 'West'),
    ('Villanova',         8,  'Utah St.',            9, 'West'),
    ('Wisconsin',         5,  'High Point',         12, 'West'),
    ('Arkansas',          4,  'Hawaii',             13, 'West'),
    ('BYU',               6,  'Texas',              11, 'West'),
    ('Gonzaga',           3,  'Kennesaw St.',       14, 'West'),
    ('Miami (FL)',        7,  'Missouri',           10, 'West'),
    ('Purdue',            2,  'Queens University',  15, 'West'),
    # MIDWEST
    ('Michigan',          1,  'Howard',             16, 'Midwest'),
    ('Georgia',           8,  'Saint Louis',         9, 'Midwest'),
    ('Texas Tech',        5,  'Akron',              12, 'Midwest'),
    ('Alabama',           4,  'Hofstra',            13, 'Midwest'),
    ('Tennessee',         6,  'SMU',                11, 'Midwest'),
    ('Virginia',          3,  'Wright St.',         14, 'Midwest'),
    ('Kentucky',          7,  'Santa Clara',        10, 'Midwest'),
    ('Iowa St.',          2,  'Tennessee St.',      15, 'Midwest'),
]

TEAM_ALIASES = {
    "St. John's":        ["St. John's (NY)", "Saint John's"],
    'Michigan St.':      ['Michigan State'],
    'North Dakota St.':  ['North Dakota State'],
    'Cal Baptist':       ['California Baptist'],
    'Ohio St.':          ['Ohio State'],
    "Saint Mary's":      ["Saint Mary's (CA)", "St. Mary's"],
    'Utah St.':          ['Utah State'],
    'Kennesaw St.':      ['Kennesaw State'],
    'Miami (FL)':        ['Miami FL'],
    'Iowa St.':          ['Iowa State'],
    'Tennessee St.':     ['Tennessee State'],
    'Wright St.':        ['Wright State'],
    'South Florida':     ['USF'],
}

tourn_teams = set()
for t_a, _, t_b, _, _ in TOURNAMENT_MATCHUPS:
    tourn_teams.add(t_a)
    tourn_teams.add(t_b)
    for a in TEAM_ALIASES.get(t_a, []): tourn_teams.add(a)
    for a in TEAM_ALIASES.get(t_b, []): tourn_teams.add(a)

stats['in_tournament'] = stats['team'].isin(tourn_teams)

# Now that in_tournament is defined, attach it to ae and compare
for frame in [ae, compare]:
    if 'in_tournament' not in frame.columns:
        frame['in_tournament'] = frame['team'].isin(tourn_teams)

stats.to_csv('race_to_10_team_stats.csv', index=False)
print(f'Saved race_to_10_team_stats.csv ({len(stats):,} teams)')
print(f'Tournament teams found in data: {stats["in_tournament"].sum()}')

### Preview: Tournament Teams — Actual vs Expected & Key Stats

In [ ]:
tourn_preview = stats[stats['in_tournament']].merge(
    ae[['team','n_games','actual_wr','expected_wr','diff']],
    on='team', how='left'
).sort_values('diff', ascending=False)

cols = ['team','games','race_win_pct','actual_wr','expected_wr','diff',
        'avg_t10_secs','blitz_rate','adj_net']
tourn_preview[[c for c in cols if c in tourn_preview.columns]]

## 11. Tournament Matchup Projections

All games are neutral site. Spread estimated from `adj_net_A - adj_net_B`.  
EV picks account for upset bonus in contest scoring: correct upset pick = `1 + (seed_diff / 4)` pts.

In [ ]:
M = 20  # prior strength

# ── Actual FanDuel lines (3/16/2026) — spread from team_a perspective ─────────
# positive = team_a favored, negative = team_a is the underdog
LINES_OVERRIDE = {
    # EAST
    ('Duke',            'Siena'):                  +28.5,
    ('Ohio St.',        'TCU'):                    +2.5,
    ("St. John's",      'Northern Iowa'):          +10.5,
    ('Kansas',          'Cal Baptist'):            +14.5,
    ('Louisville',      'South Florida'):          +5.5,
    ('Michigan St.',    'North Dakota St.'):       +15.5,
    ('UCLA',            'UCF'):                    +5.5,
    ('UConn',           'Furman'):                 +20.5,
    # SOUTH
    # Florida vs Lehigh — no line yet, fall back to adj_net
    ('Clemson',         'Iowa'):                   -2.5,   # Iowa favored
    ('Vanderbilt',      'McNeese'):                +11.5,
    ('Nebraska',        'Troy'):                   +13.5,
    ('North Carolina',  'VCU'):                    +2.5,
    ('Illinois',        'Penn'):                   +23.5,
    ("Saint Mary's",    'Texas A&M'):              +3.5,
    ('Houston',         'Idaho'):                  +23.5,
    # WEST
    ('Arizona',         'Long Island University'): +31.5,
    ('Villanova',       'Utah St.'):               -2.5,   # Utah St. favored
    ('Wisconsin',       'High Point'):             +10.5,
    ('Arkansas',        'Hawaii'):                 +15.5,
    # BYU vs Texas — no line yet, fall back to adj_net
    ('Gonzaga',         'Kennesaw St.'):           +20.5,
    ('Miami (FL)',      'Missouri'):               +1.5,
    ('Purdue',          'Queens University'):      +25.5,
    # MIDWEST
    # Michigan vs Howard — no line yet, fall back to adj_net
    ('Georgia',         'Saint Louis'):            +2.5,
    ('Texas Tech',      'Akron'):                  +7.5,
    ('Alabama',         'Hofstra'):                +11.5,
    # Tennessee vs SMU — no line yet, fall back to adj_net
    ('Virginia',        'Wright St.'):             +18.5,
    ('Kentucky',        'Santa Clara'):            +3.5,
    ('Iowa St.',        'Tennessee St.'):          +24.5,
}

# Lookups
stats_lookup = stats.set_index('team').to_dict('index')
ae_lookup    = ae.set_index('team')[['diff','n_games']].to_dict('index')

def get_stat(team, field, default=np.nan):
    for name in [team] + TEAM_ALIASES.get(team, []):
        if name in stats_lookup:
            return stats_lookup[name].get(field, default)
    return default

def get_ae(team):
    for name in [team] + TEAM_ALIASES.get(team, []):
        if name in ae_lookup:
            row = ae_lookup[name]
            return float(row['diff']), int(row['n_games'])
    return 0.0, 0

def shrink(diff, n, m=M):
    return diff * (n / (n + m))


rows = []
for team_a, seed_a, team_b, seed_b, region in TOURNAMENT_MATCHUPS:
    net_a = get_stat(team_a, 'adj_net')
    net_b = get_stat(team_b, 'adj_net')

    # ── Spread: actual line if available, else adj_net diff ───────────────────
    if (team_a, team_b) in LINES_OVERRIDE:
        spread_a   = LINES_OVERRIDE[(team_a, team_b)]
        line_source = 'FanDuel'
    elif pd.notna(net_a) and pd.notna(net_b):
        spread_a   = net_a - net_b
        line_source = 'adj_net'
    else:
        spread_a   = 0.0
        line_source = 'none'

    # ── Raw model probability ─────────────────────────────────────────────────
    p_a_model = float(sigmoid(k * spread_a + b))

    # ── Bayesian adjustment ───────────────────────────────────────────────────
    diff_a, n_a = get_ae(team_a)
    diff_b, n_b = get_ae(team_b)
    adj      = (shrink(diff_a, n_a) - shrink(diff_b, n_b)) / 2
    p_a_adj  = float(np.clip(p_a_model + adj, 0.05, 0.95))
    p_b_adj  = 1.0 - p_a_adj

    # ── EV picks ──────────────────────────────────────────────────────────────
    seed_diff = abs(seed_a - seed_b)
    bonus     = seed_diff / 4.0
    if seed_a < seed_b:
        ev_a = p_a_adj * 1.0
        ev_b = p_b_adj * (1.0 + bonus)
    else:
        ev_a = p_a_adj * (1.0 + bonus)
        ev_b = p_b_adj * 1.0

    pick    = team_a if ev_a >= ev_b else team_b
    pick_ev = max(ev_a, ev_b)

    rows.append({
        'region':        region,
        'team_a':        team_a,   'seed_a':  seed_a,
        'team_b':        team_b,   'seed_b':  seed_b,
        'line_source':   line_source,
        'spread_a':      round(spread_a, 1),
        'adj_net_a':     round(net_a, 2) if pd.notna(net_a) else np.nan,
        'adj_net_b':     round(net_b, 2) if pd.notna(net_b) else np.nan,
        'p_a_model':     round(p_a_model, 3),
        'p_b_model':     round(1 - p_a_model, 3),
        'diff_a':        round(diff_a, 3),   'n_a': n_a,
        'diff_b':        round(diff_b, 3),   'n_b': n_b,
        'shrunk_adj':    round(adj, 3),
        'p_a_adj':       round(p_a_adj, 3),
        'p_b_adj':       round(p_b_adj, 3),
        'ev_a':          round(ev_a, 3),
        'ev_b':          round(ev_b, 3),
        'ev_pick':       pick,
        'ev_pick_value': round(pick_ev, 3),
        'blitz_a':       round(get_stat(team_a, 'blitz_rate'), 3),
        'blitz_b':       round(get_stat(team_b, 'blitz_rate'), 3),
        'avg_t10_a':     round(get_stat(team_a, 'avg_t10_secs'), 1),
        'avg_t10_b':     round(get_stat(team_b, 'avg_t10_secs'), 1),
    })

tourn = pd.DataFrame(rows)
tourn.to_csv('race_to_10_tournament.csv', index=False)
print(f'Saved race_to_10_tournament.csv ({len(tourn)} matchups)')
print(f'  FanDuel lines used: {(tourn["line_source"]=="FanDuel").sum()}')
print(f'  adj_net fallback:   {(tourn["line_source"]=="adj_net").sum()}')
tourn

## 12. Summary: Top EV Picks & Baseline Reference

In [ ]:
print('=== TOP EV PICKS (Race to 10) ===')
top = tourn.sort_values('ev_pick_value', ascending=False).head(12)
for _, row in top.iterrows():
    matchup   = f"#{row['seed_a']} {row['team_a']} vs #{row['seed_b']} {row['team_b']}"
    spread_str = f"{row['spread_a']:+.1f}" if pd.notna(row['spread_a']) else 'N/A'
    move       = f"{row['shrunk_adj']:+.3f}" if pd.notna(row['shrunk_adj']) else '  n/a'
    print(f"  {row['ev_pick']:22s}  EV={row['ev_pick_value']:.3f}  "
          f"spread={spread_str}  "
          f"p_model={row['p_a_model']:.1%}  p_adj={row['p_a_adj']:.1%}  move={move}  "
          f"({row['region']}) | {matchup}")

print()
print('=== BASELINE: Model P(race win) by spread ===')
for pts in [-10, -5, -3, -1, 0, 1, 3, 5, 10]:
    p = float(sigmoid(k * pts + b))
    label = f'{abs(pts):2d}pt {"dog":4}' if pts < 0 else (f'  pick ' if pts == 0 else f'{pts:2d}pt {"fav":4}')
    print(f'  {label}: {p:.1%}')

print(f'\n(Bayesian prior strength M={M}: after {M} games, seasonal diff gets 50% weight)')

## 13. Matchup Graphic

In [ ]:
# ── Fetch team logos via ESPN CDN (uses source_id from CBBD) ──────────────────
LOGO_CACHE = 'cbbd_data/team_logos.csv'

if os.path.exists(LOGO_CACHE):
    logo_df = pd.read_csv(LOGO_CACHE)
    logo_lookup = dict(zip(logo_df['team'], logo_df['logo']))
    print(f'Loaded {len(logo_lookup)} logos from cache')
else:
    configuration = cbbd.Configuration(access_token=API_KEY)
    with cbbd.ApiClient(configuration) as api_client:
        teams_api = cbbd.TeamsApi(api_client)
        all_teams = teams_api.get_teams()

    logo_lookup = {}
    for t in all_teams:
        sid = getattr(t, 'source_id', None)
        if sid:
            logo_lookup[t.school] = f'https://a.espncdn.com/i/teamlogos/ncaa/500/{sid}.png'

    os.makedirs('cbbd_data', exist_ok=True)
    pd.DataFrame(logo_lookup.items(), columns=['team','logo']).to_csv(LOGO_CACHE, index=False)
    print(f'Fetched and cached {len(logo_lookup)} logos')

def get_logo(team):
    for name in [team] + TEAM_ALIASES.get(team, []):
        if name in logo_lookup:
            return logo_lookup[name]
    return ''

print('Logo lookup ready ✓')
print(f'Sample: {list(logo_lookup.items())[:3]}')

In [ ]:
def to_american_ml(p):
    p = max(0.01, min(0.99, float(p)))
    if p >= 0.5:
        return f'-{round((p / (1 - p)) * 100)}'
    else:
        return f'+{round(((1 - p) / p) * 100)}'

def fmt_diff(v):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return '<span style="color:#64748b">—</span>'
    sign = '+' if v > 0 else ''
    color = '#4ade80' if v > 0.02 else ('#f87171' if v < -0.02 else '#94a3b8')
    return f'<span style="color:{color};font-weight:600">{sign}{v:.3f}</span>'

def fmt_pct(v):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return '<span style="color:#64748b">—</span>'
    return f'{v:.1%}'

def team_cell(team, seed, logo_url):
    logo_html = (f'<img src="{logo_url}" style="width:28px;height:28px;object-fit:contain;'
                 f'vertical-align:middle;margin-right:7px" onerror="this.style.display=\'none\'">')
    return (f'<div style="display:flex;align-items:center;white-space:nowrap">'
            f'{logo_html}'
            f'<div><div style="font-size:10px;color:#64748b;letter-spacing:.5px">#{seed}</div>'
            f'<div style="color:#f1f5f9;font-weight:600;font-size:13px;line-height:1.1">{team}</div>'
            f'</div></div>')

ae_lookup_full = ae.set_index('team').to_dict('index')
def get_ae_stat(team, field):
    for name in [team] + TEAM_ALIASES.get(team, []):
        if name in ae_lookup_full:
            v = ae_lookup_full[name].get(field, None)
            return float(v) if v is not None and not (isinstance(v, float) and np.isnan(v)) else None
    return None

REGION_COLORS = {
    'East':    ('#1a3a5c', '#3b82f6'),
    'South':   ('#5c1a1a', '#ef4444'),
    'West':    ('#1a5c2e', '#22c55e'),
    'Midwest': ('#3d1a5c', '#a855f7'),
}

OUTPUT_FILE = 'race_to_10_bracket.html'
body_html = ''

for region in ['East', 'South', 'West', 'Midwest']:
    bg_dark, accent = REGION_COLORS[region]
    body_html += (
        f'<tr style="background:{bg_dark}">'
        f'<td colspan="9" style="padding:8px 14px;font-size:12px;font-weight:700;'
        f'letter-spacing:2px;color:{accent};text-transform:uppercase;border-bottom:2px solid {accent}">'
        f'{region} Region</td></tr>\n'
    )

    for row in tourn[tourn['region'] == region].itertuples():
        logo_a = get_logo(row.team_a)
        logo_b = get_logo(row.team_b)

        spr = f'{row.spread_a:+.1f}' if pd.notna(row.spread_a) else 'N/A'
        src_badge = f'<span style="font-size:9px;color:#475569;margin-left:4px">({row.line_source})</span>'

        # Fav model ML
        fav_p_model = row.p_a_model if row.spread_a >= 0 else row.p_b_model
        fav_ml_model = to_american_ml(fav_p_model)

        # Bayesian adjusted MLs — fav in amber, dog in slate
        ml_a_adj = to_american_ml(row.p_a_adj)
        ml_b_adj = to_american_ml(row.p_b_adj)
        a_is_fav = row.spread_a >= 0
        color_a = '#f59e0b' if a_is_fav else '#94a3b8'
        color_b = '#94a3b8' if a_is_fav else '#f59e0b'

        # Bayesian move: shrunk_adj is the adjustment to p_a
        # positive = moved towards team_a, negative = moved towards team_b
        adj = float(row.shrunk_adj) if pd.notna(row.shrunk_adj) else 0.0
        if abs(adj) < 0.001:
            move_html = '<span style="font-size:10px;color:#334155">no move</span>'
        else:
            moved_towards = row.team_a if adj > 0 else row.team_b
            moved_logo    = logo_a if adj > 0 else logo_b
            pct_str       = f'+{abs(adj)*100:.1f}%'
            move_html = (
                f'<div style="display:flex;align-items:center;gap:5px;margin-top:5px;'
                f'padding-top:5px;border-top:1px solid #334155">'
                f'<img src="{moved_logo}" style="width:16px;height:16px;object-fit:contain" '
                f'onerror="this.style.display=\'none\'">'
                f'<span style="font-size:11px;color:#38bdf8;font-weight:600">{pct_str}</span>'
                f'<span style="font-size:10px;color:#475569">toward {moved_towards}</span>'
                f'</div>'
            )

        bayes_cell = (
            f'<div style="line-height:1.5">'
            f'<span style="font-size:12px;font-weight:700;color:{color_a}">{ml_a_adj}</span>'
            f'<span style="font-size:10px;color:#475569;margin-left:4px">{row.team_a}</span>'
            f'<br>'
            f'<span style="font-size:12px;font-weight:700;color:{color_b}">{ml_b_adj}</span>'
            f'<span style="font-size:10px;color:#475569;margin-left:4px">{row.team_b}</span>'
            f'{move_html}'
            f'</div>'
        )

        act_a  = get_ae_stat(row.team_a, 'actual_wr')
        diff_a = get_ae_stat(row.team_a, 'diff')
        act_b  = get_ae_stat(row.team_b, 'actual_wr')
        diff_b = get_ae_stat(row.team_b, 'diff')

        body_html += (
            f'<tr style="background:#1e293b;border-bottom:1px solid #1e293b">'
            f'<td style="padding:9px 12px">{team_cell(row.team_a, row.seed_a, logo_a)}</td>'
            f'<td style="padding:9px 12px">{team_cell(row.team_b, row.seed_b, logo_b)}</td>'
            f'<td style="padding:9px 12px;text-align:center;font-size:13px;color:#e2e8f0">{spr}{src_badge}</td>'
            f'<td style="padding:9px 12px;text-align:center;font-size:14px;font-weight:700;color:#f59e0b">{fav_ml_model}</td>'
            f'<td style="padding:9px 12px;text-align:center;font-size:13px;color:#cbd5e1">{fmt_pct(act_a)}</td>'
            f'<td style="padding:9px 12px;text-align:center;font-size:13px;color:#cbd5e1">{fmt_pct(act_b)}</td>'
            f'<td style="padding:9px 12px;text-align:center;font-size:12px">{fmt_diff(diff_a)}</td>'
            f'<td style="padding:9px 12px;text-align:center;font-size:12px">{fmt_diff(diff_b)}</td>'
            f'<td style="padding:9px 12px;text-align:left">{bayes_cell}</td>'
            f'</tr>\n'
        )

html = f'''<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8">
<title>Race to 10 — 2026 NCAA Tournament</title>
<style>
  * {{ box-sizing: border-box; margin: 0; padding: 0; }}
  body {{ font-family: "Segoe UI", "Inter", Arial, sans-serif; background: #0f172a; color: #e2e8f0; padding: 28px 20px; }}
  h1 {{ text-align: center; font-size: 22px; font-weight: 800; color: #f8fafc; margin-bottom: 4px; letter-spacing: 1px; text-transform: uppercase; }}
  .subtitle {{ text-align: center; color: #475569; font-size: 12px; margin-bottom: 18px; }}
  .legend {{ display: flex; gap: 18px; justify-content: center; margin-bottom: 20px; font-size: 11px; color: #64748b; }}
  .legend span {{ display: flex; align-items: center; gap: 5px; }}
  .dot {{ width: 9px; height: 9px; border-radius: 50%; }}
  table {{ width: 100%; border-collapse: collapse; background: #151e2d; border: 1px solid #1e293b; border-radius: 8px; overflow: hidden; }}
  thead th {{ background: #0f172a; padding: 10px 12px; font-size: 10px; font-weight: 700; text-transform: uppercase; letter-spacing: 1px; color: #64748b; border-bottom: 2px solid #334155; white-space: nowrap; text-align: center; }}
  thead th:first-child, thead th:nth-child(2) {{ text-align: left; }}
  tbody tr:hover {{ background: #263548 !important; }}
  td {{ border-top: 1px solid #1e3a5f20; vertical-align: middle; }}
</style>
</head>
<body>
<h1>Race to 10 — 2026 NCAA Tournament</h1>
<p class="subtitle">Logistic model fit on 2025-26 season · Bayesian shrinkage M=20 · FanDuel spreads (adj_net fallback)</p>
<div class="legend">
  <span><span class="dot" style="background:#f59e0b"></span>Spread favorite</span>
  <span><span class="dot" style="background:#38bdf8"></span>Bayesian move direction</span>
  <span><span class="dot" style="background:#4ade80"></span>Overperformer vs spread</span>
  <span><span class="dot" style="background:#f87171"></span>Underperformer vs spread</span>
</div>
<table>
  <thead>
    <tr>
      <th style="text-align:left">Team 1</th>
      <th style="text-align:left">Team 2</th>
      <th>FanDuel Spread<br><span style="font-weight:400;color:#475569">(T1 persp.)</span></th>
      <th>Fav Model<br>Race-to-10 ML</th>
      <th>T1 Race<br>Win %</th>
      <th>T2 Race<br>Win %</th>
      <th>T1 vs<br>Expected</th>
      <th>T2 vs<br>Expected</th>
      <th>Bayes Adj.<br>Race-to-10 ML</th>
    </tr>
  </thead>
  <tbody>
{body_html}  </tbody>
</table>
</body>
</html>'''

with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    f.write(html)

print(f'Saved {OUTPUT_FILE}')
